# Quick Demo: Deception-Specific SAE Analysis (~15 min)

Streamlined version running **Experiment 1** (SAE comparison) and **Experiment 3** (aggregate analysis).
For the full 6-experiment pipeline, see `colab_deception_sae_research.ipynb`.

**Reference**: DeLeeuw, Chawla et al. "The Secret Agenda"

**Before you start**: Enable T4 GPU (Runtime -> Change runtime type -> T4 GPU)

In [ ]:
# 1. Setup (~2 min)
import torch, subprocess
assert torch.cuda.is_available(), "Enable T4 GPU: Runtime -> Change runtime type -> T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
%%bash
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>/dev/null
source "$HOME/.cargo/env"
[ ! -d "nanochat-SAE" ] && git clone https://github.com/SolshineCode/nanochat-SAE.git || (cd nanochat-SAE && git pull)
pip install -q datasets tiktoken tokenizers huggingface_hub tqdm matplotlib maturin psutil regex seaborn scikit-learn
cd nanochat-SAE/rustbpe && maturin develop --release 2>&1 | tail -1
echo "Done!"

In [ ]:
# 2. Load model (~2 min)
import sys, os, gc, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

sys.path.insert(0, '/content/nanochat-SAE')
os.chdir('/content/nanochat-SAE')

from huggingface_hub import hf_hub_download
MODEL_DIR = Path('/content/nanochat-d32')
MODEL_DIR.mkdir(exist_ok=True)
TOK_DIR = Path('/content/nanochat-tokenizer')
TOK_DIR.mkdir(exist_ok=True)

for fn in ['model_000650.pt', 'meta_000650.json']:
    hf_hub_download(repo_id='karpathy/nanochat-d32', filename=fn, local_dir=MODEL_DIR)
for fn in ['token_bytes.pt', 'tokenizer.pkl']:
    hf_hub_download(repo_id='karpathy/nanochat-d32', filename=fn, local_dir=TOK_DIR)

from nanochat.gpt import GPT, GPTConfig
from nanochat.tokenizer import RustBPETokenizer

checkpoint = torch.load(MODEL_DIR / 'model_000650.pt', map_location='cpu', mmap=True)
state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
wte = state_dict.get('transformer.wte.weight')
vocab_size, n_embd = wte.shape
n_layer = max(int(k.split('.')[2]) for k in state_dict if 'transformer.h.' in k) + 1

model_config = GPTConfig(
    vocab_size=int(vocab_size), n_embd=int(n_embd), n_layer=n_layer,
    n_head=int(n_embd)//128, n_kv_head=int(n_embd)//128, sequence_len=2048,
)
model = GPT(model_config)
model.load_state_dict(state_dict, strict=False)
del checkpoint, state_dict; gc.collect()
model = model.to(device='cuda', dtype=torch.bfloat16).eval()

tokenizer = RustBPETokenizer.from_directory(str(TOK_DIR))
print(f"Model: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params, GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# 3. Generate deception dataset + collect activations (~3 min)
from sae.deception_data import DeceptionPromptGenerator, DeceptionDataset, LabeledActivationCollector

generator = DeceptionPromptGenerator(seed=42)
prompts = generator.generate(n_per_category=200)  # Smaller for quick demo
dataset = DeceptionDataset(prompts)
stats = generator.get_statistics(prompts)
print(f"Generated {stats['total']} prompts: {stats['category_counts']}")

# Collect from layer 16
LAYER = 16
HP = f'blocks.{LAYER}.hook_resid_post'

collector = LabeledActivationCollector(
    model=model, hook_points=[HP], device='cpu', decision_token_offset=0,
)
with collector:
    collector.collect_from_prompts(prompts, tokenizer, max_seq_len=512, verbose=True)

all_acts = collector.get_activations()[HP]
categories = [m.category for m in collector.metadata_list]

acts_by_cat = {}
for cat in ['deceptive', 'honest', 'contradiction', 'neutral']:
    mask = [i for i, c in enumerate(categories) if c == cat]
    if mask: acts_by_cat[cat] = all_acts[mask]

print(f"\nActivations: {all_acts.shape}")
for cat, a in acts_by_cat.items(): print(f"  {cat}: {a.shape[0]}")

In [ ]:
# 4. Experiment 1: Train deceptive vs mixed SAE (~5 min)
from sae.config import SAEConfig
from sae.models import TopKSAE
from sae.trainer import SAETrainer
from sae.deception_eval import DeceptionEvaluator

D_IN = model_config.n_embd

def quick_train(activations, name):
    config = SAEConfig(
        d_in=D_IN, expansion_factor=4, activation='topk', k=32,
        hook_point=HP, batch_size=128, learning_rate=3e-4,
        eval_every=9999999, save_every=9999999, resample_interval=9999999,
    )
    sae = TopKSAE(config).cuda()
    act_mean = activations.mean(dim=0)
    act_std = activations.std(dim=0).clamp(min=1e-6)
    acts_norm = (activations - act_mean) / act_std
    n_val = max(20, len(acts_norm) // 10)
    trainer = SAETrainer(sae=sae, config=config, activations=acts_norm[n_val:],
                        val_activations=acts_norm[:n_val], device='cuda')
    for epoch in range(3):
        m = trainer.train_epoch(verbose=False)
        print(f"  {name} epoch {epoch+1}: loss={m['total_loss']:.6f}")
    sae.cpu(); torch.cuda.empty_cache()
    return sae, config

print("Training deception SAE...")
sae_dec, _ = quick_train(acts_by_cat['deceptive'], 'deceptive')
print("\nTraining mixed SAE...")
sae_mix, _ = quick_train(all_acts, 'mixed')

# Compute discriminability
evaluator = DeceptionEvaluator(device='cpu')
dec_acts, hon_acts = acts_by_cat['deceptive'], acts_by_cat['honest']

for name, sae in [('deceptive', sae_dec), ('mixed', sae_mix)]:
    d = evaluator.compute_discriminability(sae, dec_acts, hon_acts)
    _, scores = evaluator.get_top_discriminative_features(d, top_k=10)
    print(f"\n{name} SAE: top-10 mean |d| = {scores.mean():.3f}")

In [ ]:
# 5. Experiment 3: Aggregate analysis (~3 min)
from sae.deception_eval import run_full_evaluation

results = run_full_evaluation(
    sae=sae_mix, activations_by_category=acts_by_cat,
    labels=categories, all_activations=all_acts,
)

print("=== Aggregate Analysis ===")
print(f"Binary probe: acc={results['linear_probe_binary']['accuracy']:.3f}, AUC={results['linear_probe_binary']['auc_roc']:.3f}")
print(f"4-class probe: acc={results['linear_probe_full']['accuracy']:.3f}")
print(f"Cluster purity: {results['cluster_purity_binary']['purity']:.3f}")
print(f"t-SNE silhouette: {results['tsne']['silhouette_score']:.3f}")
print(f"Cosine divergence: {results['cosine_divergence']}")

In [ ]:
# 6. Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

color_map = {'deceptive': 'red', 'honest': 'blue', 'contradiction': 'orange', 'neutral': 'gray'}

# t-SNE
emb = results['tsne']['embedding']
for cat in ['neutral', 'contradiction', 'honest', 'deceptive']:
    mask = [i for i, l in enumerate(results['tsne']['labels']) if l == cat]
    if mask:
        axes[0].scatter(emb[mask, 0], emb[mask, 1], c=color_map[cat], label=cat, alpha=0.5, s=15)
axes[0].set_title(f"t-SNE (silhouette={results['tsne']['silhouette_score']:.3f})")
axes[0].legend(fontsize=8)

# PCA
emb_pca = results['pca']['embedding']
for cat in ['neutral', 'contradiction', 'honest', 'deceptive']:
    mask = [i for i, l in enumerate(results['pca']['labels']) if l == cat]
    if mask:
        axes[1].scatter(emb_pca[mask, 0], emb_pca[mask, 1], c=color_map[cat], label=cat, alpha=0.5, s=15)
axes[1].set_title(f"PCA (separability={results['pca']['linear_separability']:.3f})")
axes[1].legend(fontsize=8)

# Discriminability comparison
d_dec = evaluator.compute_discriminability(sae_dec, dec_acts, hon_acts)
d_mix = evaluator.compute_discriminability(sae_mix, dec_acts, hon_acts)
axes[2].hist(d_dec.abs().numpy(), bins=50, alpha=0.5, label='Deception SAE', color='coral')
axes[2].hist(d_mix.abs().numpy(), bins=50, alpha=0.5, label='Mixed SAE', color='steelblue')
axes[2].set_xlabel("|Cohen's d|")
axes[2].set_ylabel('Feature count')
axes[2].set_title('Feature Discriminability')
axes[2].legend()

plt.tight_layout()
plt.savefig('/content/deception_quick_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Done! Full results in colab_deception_sae_research.ipynb")